# AI-ITI Study Assistant — RAG Pipeline

**Student:** Ali Ezz Ali  
**Track:** Core Track — text RAG  
**Runtime:** GitHub Codespaces (cloud), Python 3.12, Chroma, Ollama

This notebook builds and evaluates a Retrieval-Augmented Generation pipeline over twelve
cleaned AI-ITI lab documents. It is designed to run from top to bottom in the repository's
cloud environment.

## 1. Load and inspect

The corpus contains one Markdown document per solved lab. Notebook outputs and embedded
images were removed during conversion; explanatory markdown and relevant code were kept.
No OCR is required because all selected files are text-extractable notebooks.

In [1]:
import json
import os
import sys
from pathlib import Path

import chromadb
import pandas as pd
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DOCUMENTS = ROOT / "data" / "source_documents"
VECTOR_STORE = ROOT / "data" / "vector_store"
QUESTIONS_FILE = ROOT / "data" / "evaluation" / "questions.json"
documents = sorted(DOCUMENTS.glob("*.md"))
inspection = pd.DataFrame({
    "document": [path.name for path in documents],
    "format": [path.suffix for path in documents],
    "characters": [len(path.read_text(encoding="utf-8")) for path in documents],
    "needs_ocr": [False] * len(documents),
})
print(f"Documents: {len(documents)}")
print(f"Total characters: {inspection['characters'].sum():,}")
inspection

Documents: 12
Total characters: 117,674


,document,format,characters,needs_ocr
0,LAB-01-OOP.md,.md,7051,False
1,LAB-02-Advanced-Python.md,.md,6686,False
2,LAB-03-Computer-Vision.md,.md,15929,False
3,LAB-04-Deep-Vision.md,.md,14071,False
4,LAB-05-Object-Detection.md,.md,17003,False
5,LAB-06-YOLO.md,.md,21069,False
6,LAB-07-Transfer-Learning.md,.md,5651,False
7,LAB-08-NLP-Preprocessing.md,.md,3354,False
8,LAB-09-Sentiment-POS-NER.md,.md,2497,False
9,LAB-10-Topic-Modeling.md,.md,7943,False


## 2. Chunking strategy

Documents are split by Markdown section, then into **320-word chunks with a 60-word
overlap**. Section boundaries preserve topic meaning and the overlap protects explanations
that cross a window boundary. A 320-word window is large enough for code-plus-explanation
context but small enough to keep retrieval focused. Every chunk stores its source document,
section heading, and chunk number for citations.

In [2]:
from scripts.build_vector_store import load_chunks

chunks = load_chunks(DOCUMENTS, size=320, overlap=60)
chunk_stats = pd.Series([len(chunk.text.split()) for chunk in chunks]).describe()
print(f"Chunks: {len(chunks)}")
chunk_stats

Chunks: 264


count    264.000000
mean      62.026515
std       66.960485
min        1.000000
25%       26.000000
50%       41.500000
75%       73.250000
max      320.000000
dtype: float64

## 3. Embeddings and persistent vector store

The multilingual `paraphrase-multilingual-MiniLM-L12-v2` sentence-transformer creates
embeddings for English and Arabic questions. Chroma stores the vectors on disk so the API
loads them at startup instead of rebuilding them per request.

In [3]:
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
COLLECTION_NAME = "ai_iti_course"
embedding_function = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)
client = chromadb.PersistentClient(path=str(VECTOR_STORE))
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = client.get_or_create_collection(
    COLLECTION_NAME,
    embedding_function=embedding_function,
    metadata={"hnsw:space": "cosine"},
)
collection.add(
    ids=[f"{chunk.source}:{chunk.chunk_id}" for chunk in chunks],
    documents=[chunk.text for chunk in chunks],
    metadatas=[{
        "source": chunk.source,
        "section": chunk.section,
        "chunk_id": chunk.chunk_id,
    } for chunk in chunks],
)
print(f"Persisted {collection.count()} chunks to {VECTOR_STORE}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Persisted 264 chunks to /home/runner/work/ai-iti-rag-assistant/ai-iti-rag-assistant/data/vector_store


## 4. Retrieval and citation grounding

In [4]:
def retrieve(question: str, top_k: int = 4) -> list[dict]:
    result = collection.query(
        query_texts=[question],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {
            "text": text,
            "source": metadata["source"],
            "section": metadata["section"],
            "chunk_id": metadata["chunk_id"],
            "distance": round(float(distance), 4),
        }
        for text, metadata, distance in zip(
            result["documents"][0],
            result["metadatas"][0],
            result["distances"][0],
        )
    ]

questions = json.loads(QUESTIONS_FILE.read_text(encoding="utf-8"))
retrieval_rows = []
for item in questions:
    results = retrieve(item["question"])
    sources = [result["source"] for result in results]
    retrieval_rows.append({
        "question": item["question"],
        "expected_source": item["expected_source"],
        "top_source": sources[0],
        "retrieved_sources": ", ".join(sources),
        "relevant": item["expected_source"] in sources,
    })
retrieval_df = pd.DataFrame(retrieval_rows)
retrieval_df

,question,expected_source,top_source,retrieved_sources,relevant
0,What are inheritance and polymorphism in Pytho...,LAB-01-OOP,LAB-01-OOP,"LAB-01-OOP, LAB-01-OOP, LAB-01-OOP, LAB-01-OOP",True
1,How does a Python generator differ from a norm...,LAB-02-Advanced-Python,LAB-12-Transformers,"LAB-12-Transformers, LAB-02-Advanced-Python, L...",True
2,What steps does Canny edge detection use?,LAB-03-Computer-Vision,LAB-05-Object-Detection,"LAB-05-Object-Detection, LAB-03-Computer-Visio...",True
3,Why do residual connections help a ResNet train?,LAB-04-Deep-Vision,LAB-03-Computer-Vision,"LAB-03-Computer-Vision, LAB-04-Deep-Vision, LA...",True
4,How does Non-Max Suppression remove duplicate ...,LAB-05-Object-Detection,LAB-05-Object-Detection,"LAB-05-Object-Detection, LAB-05-Object-Detecti...",True
5,What are the main stages of the YOLO mini-proj...,LAB-06-YOLO,LAB-06-YOLO,"LAB-06-YOLO, LAB-06-YOLO, LAB-06-YOLO, LAB-12-...",True
6,Why is the pretrained base model frozen before...,LAB-07-Transfer-Learning,LAB-07-Transfer-Learning,"LAB-07-Transfer-Learning, LAB-04-Deep-Vision, ...",True
7,What is the difference between stemming and le...,LAB-08-NLP-Preprocessing,LAB-08-NLP-Preprocessing,"LAB-08-NLP-Preprocessing, LAB-12-Transformers,...",True
8,How are POS tagging and named entity recogniti...,LAB-09-Sentiment-POS-NER,LAB-09-Sentiment-POS-NER,"LAB-09-Sentiment-POS-NER, LAB-08-NLP-Preproces...",True
9,How do LDA and LSA differ in topic modeling?,LAB-10-Topic-Modeling,LAB-10-Topic-Modeling,"LAB-10-Topic-Modeling, LAB-06-YOLO, LAB-11-Ara...",True


In [5]:
retrieval_accuracy = retrieval_df["relevant"].mean()
print(f"Retrieval hit rate @4: {retrieval_accuracy:.1%}")

Retrieval hit rate @4: 100.0%


## 5. Prompt template

The generation prompt explicitly limits Ollama to retrieved evidence, numbers every source,
requires inline citations, and provides a refusal sentence for insufficient context. This is
the primary hallucination control.

In [6]:
from backend.app.services.generation import SYSTEM_PROMPT
print(SYSTEM_PROMPT)

You are the AI-ITI Study Assistant.
Answer only from the supplied course context. Never use unsupported outside knowledge.
If the context is insufficient, say: "I do not have enough information in the course material."
Be concise but educational. Cite factual statements using [Source N].
Do not invent file names, sections, metrics, or citations.


## 6. End-to-end evaluation with Ollama

The following evaluation runs all test questions through retrieval and `qwen2.5:1.5b`.
`retrieval_correct` checks whether the expected lab is in the top four results.
`citation_grounded` checks whether the answer uses the required source markers. Final manual
review should inspect whether each statement is actually supported by its retrieved text.

In [7]:
from backend.app.core.config import Settings
from backend.app.services.generation import GenerationService
from backend.app.services.retrieval import RetrievedChunk

generator = GenerationService(Settings())
evaluation_rows = []
for item in questions:
    retrieved = retrieve(item["question"])
    service_chunks = [RetrievedChunk(
        text=result["text"],
        source=result["source"],
        section=result["section"],
        chunk_id=result["chunk_id"],
        distance=result["distance"],
    ) for result in retrieved]
    answer = generator.answer(item["question"], service_chunks)
    sources = [result["source"] for result in retrieved]
    evaluation_rows.append({
        "question": item["question"],
        "retrieved_source": sources[0],
        "answer": answer,
        "retrieval_correct": item["expected_source"] in sources,
        "citation_grounded": "[Source" in answer,
    })
evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df

,question,retrieved_source,answer,retrieval_correct,citation_grounded
0,What are inheritance and polymorphism in Pytho...,LAB-01-OOP,Inheritance and polymorphism are fundamental c...,True,True
1,How does a Python generator differ from a norm...,LAB-12-Transformers,A Python generator differs from a normal funct...,True,True
2,What steps does Canny edge detection use?,LAB-05-Object-Detection,Canny edge detection uses the following steps:...,True,True
3,Why do residual connections help a ResNet train?,LAB-03-Computer-Vision,Residual connections help a ResNet train by st...,True,True
4,How does Non-Max Suppression remove duplicate ...,LAB-05-Object-Detection,Non-Max Suppression (NMS) removes duplicate de...,True,True
5,What are the main stages of the YOLO mini-proj...,LAB-06-YOLO,The main stages of the YOLO mini-project are:\...,True,True
6,Why is the pretrained base model frozen before...,LAB-07-Transfer-Learning,The pretrained base model is frozen before fin...,True,True
7,What is the difference between stemming and le...,LAB-08-NLP-Preprocessing,Stemming and lemmatization are both text-norma...,True,True
8,How are POS tagging and named entity recogniti...,LAB-09-Sentiment-POS-NER,POS tagging and named entity recognition are d...,True,True
9,How do LDA and LSA differ in topic modeling?,LAB-10-Topic-Modeling,LDA (Latent Dirichlet Allocation) and LSA (Lat...,True,True


In [8]:
summary = pd.DataFrame({
    "metric": ["Questions", "Retrieval hit rate @4", "Answers with citations"],
    "value": [
        len(evaluation_df),
        f"{evaluation_df['retrieval_correct'].mean():.1%}",
        f"{evaluation_df['citation_grounded'].mean():.1%}",
    ],
})
evaluation_path = ROOT / "data" / "evaluation" / "results.csv"
evaluation_df.to_csv(evaluation_path, index=False)
summary

,metric,value
0,Questions,12
1,Retrieval hit rate @4,100.0%
2,Answers with citations,100.0%


## 7. Failure analysis and mitigation

Likely failures include overlapping terminology across labs, questions that are broader than
one section, and a small CPU-friendly LLM producing terse answers. Mitigations are section-aware
chunks, overlap, top-4 retrieval, explicit source metadata, low-temperature generation, a strict
context-only prompt, and an insufficient-context refusal. Retrieval misses should be handled by
improving document headings or adding a reranker—not by allowing unsupported model knowledge.

## 8. Export

The vector store is persisted in `data/vector_store/`; configuration is recorded in `.env.example`.
The FastAPI lifespan loads this existing store once at startup. No indexing occurs during a query.

In [9]:
assert (VECTOR_STORE / "chroma.sqlite3").exists()
assert collection.count() == len(chunks)
print("Export verified:", VECTOR_STORE)
print("The backend can load this store without rebuilding it.")

Export verified: /home/runner/work/ai-iti-rag-assistant/ai-iti-rag-assistant/data/vector_store
The backend can load this store without rebuilding it.
